# EQO notebook: evolution readiness

This is an end-to-end multi-tool example: **OpenQEvo → QASMTrans → STABSim and NWQEC**. The Hamiltonian is created as a typed EQO input; all four operations run through their published workflow and declared workers.

In [ ]:
import json
import os

from eqo import EQOClient, render_artifact, render_run

eqo = EQOClient.connect(os.environ.get("EQO_ENDPOINT", "http://127.0.0.1:8080"))
eqo.health()

## Create a Pauli-Hamiltonian artifact

The terms are small enough for this example, but the workflow still records exactly which input, synthesis parameters, and runtime revisions produced the circuit.

In [ ]:
hamiltonian = {
    "qubits": 2,
    "terms": [
        {"pauli": "ZI", "coefficient": 1.0},
        {"pauli": "IZ", "coefficient": 0.5},
        {"pauli": "XX", "coefficient": 0.25},
    ],
}
input_hamiltonian = eqo.artifacts.create_input(
    "qhpc.pauli-hamiltonian@1",
    json.dumps(hamiltonian),
    name="two-qubit-evolution.json",
)
input_hamiltonian.metadata

## Submit the composed workflow

The workflow fixes a reviewed second-order Trotter configuration. It fans the generated circuit out to QASMTrans and NWQEC, then sends the transpiled circuit to STABSim.

In [ ]:
workflow = next((item for item in eqo.workflows.list() if item["id"] == "showcase-evolution-readiness"), None)
if workflow is None:
    raise RuntimeError("Restart EQO Local to publish showcase-evolution-readiness.")
run = eqo.workflows.submit(
    workflow["id"], workflow["version"],
    inputs={"hamiltonian": input_hamiltonian.id},
    execution_target="development-slurm-docker",
)
render_run(run)

In [ ]:
completed = run.wait(timeout=600)
if completed.state != "succeeded":
    raise RuntimeError(f"Evolution workflow ended in {completed.state}; inspect render_run(completed).")
render_run(completed)

## Inspect each handoff

These are independent, checksum-verified run artifacts—not values copied through notebook memory.

In [ ]:
for artifact_type in (
    "qhpc.evolution-synthesis-report@1",
    "qhpc.transpiled-circuit@1",
    "qhpc.circuit-metrics@1",
    "qhpc.clifford-t-counts@1",
):
    display(render_artifact(completed.artifacts.by_type(artifact_type)))